In [3]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata
import tarfile

login(token=userdata.get('HF_TOKEN'))

path = hf_hub_download(
    repo_id="HashimAbbasii/asad-spark-logs",
    filename="spark_logs.tar.gz",
    repo_type="dataset"
)

with tarfile.open(path) as tar:
    tar.extractall("/content/spark_data")

print("Extraction done!")

/tmp/ipykernel_4249/4197809705.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content/spark_data")


Extraction done!


In [4]:
!pip install drain3 -q
print("Setup done!")

Setup done!


In [5]:
import os

base_path = "/content/spark_data"
log_files = []

for root, dirs, files in os.walk(base_path):
    for f in files:
        if f.endswith(".log"):
            log_files.append(os.path.join(root, f))

print(f"Total log files found: {len(log_files)}")
print("Sample:", log_files[0])

Total log files found: 3852
Sample: /content/spark_data/application_1485248649253_0124/container_1485248649253_0124_02_000014.log


In [1]:
import os
import pandas as pd

# Step 1: log files collect karo
base_path = "/content/spark_data"
log_files = []

for root, dirs, files in os.walk(base_path):
    for f in files:
        if f.endswith(".log"):
            log_files.append(os.path.join(root, f))

print(f"Total log files found: {len(log_files)}")

# Step 2: Output folder
import os
os.makedirs("/content/spark_data_processed", exist_ok=True)
output_csv = "/content/spark_data_processed/spark_logs_raw.csv"

# Step 3: Chunked write - memory mein load nahi, seedha file mein
CHUNK_SIZE = 200  # 200 files ek baar process karo

header_written = False

for i in range(0, len(log_files), CHUNK_SIZE):
    chunk = log_files[i:i+CHUNK_SIZE]
    records = []

    for filepath in chunk:
        parts = filepath.split("/")
        app_id = parts[-2]
        container_id = parts[-1].replace(".log", "")

        try:
            with open(filepath, "r", errors="ignore") as f:
                lines = f.readlines()
        except Exception as e:
            continue

        for line in lines:
            line = line.strip()
            if line:
                records.append({
                    "app_id": app_id,
                    "container_id": container_id,
                    "raw_line": line
                })

    # Chunk ko CSV mein append karo
    chunk_df = pd.DataFrame(records)
    chunk_df.to_csv(output_csv,
                    mode='a',
                    header=not header_written,
                    index=False)
    header_written = True

    print(f"Processed {min(i+CHUNK_SIZE, len(log_files))}/{len(log_files)} files...")
    del records, chunk_df  # memory free karo

print("Done! CSV saved at:", output_csv)

# Step 4: Verify
df_check = pd.read_csv(output_csv, nrows=5)
print(df_check)

Total log files found: 3852
Processed 200/3852 files...
Processed 400/3852 files...
Processed 600/3852 files...
Processed 800/3852 files...
Processed 1000/3852 files...
Processed 1200/3852 files...
Processed 1400/3852 files...
Processed 1600/3852 files...
Processed 1800/3852 files...
Processed 2000/3852 files...
Processed 2200/3852 files...
Processed 2400/3852 files...
Processed 2600/3852 files...
Processed 2800/3852 files...
Processed 3000/3852 files...
Processed 3200/3852 files...
Processed 3400/3852 files...
Processed 3600/3852 files...
Processed 3800/3852 files...
Processed 3852/3852 files...
Done! CSV saved at: /content/spark_data_processed/spark_logs_raw.csv
                           app_id                            container_id  \
0  application_1485248649253_0124  container_1485248649253_0124_02_000014   
1  application_1485248649253_0124  container_1485248649253_0124_02_000014   
2  application_1485248649253_0124  container_1485248649253_0124_02_000014   
3  application_1485

In [2]:
import pandas as pd
import os
import re

# CSV info check
csv_path = "/content/spark_data_processed/spark_logs_raw.csv"
file_size = os.path.getsize(csv_path) / (1024*1024)
print(f"CSV size: {file_size:.1f} MB")

# Total rows count (bina load kiye)
row_count = sum(1 for _ in open(csv_path)) - 1  # minus header
print(f"Total rows: {row_count:,}")

CSV size: 5007.9 MB
Total rows: 33,236,604


In [3]:
import pandas as pd
import os

csv_path = "/content/spark_data_processed/spark_logs_raw.csv"
output_sampled = "/content/spark_data_processed/spark_logs_sampled.csv"

CHUNK_SIZE = 100_000
SAMPLE_RATE = 0.05  # 5%

header_written = False
total_sampled = 0

for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, on_bad_lines='skip'):
    sampled = chunk.sample(frac=SAMPLE_RATE, random_state=42)

    sampled.to_csv(output_sampled,
                   mode='a',
                   header=not header_written,
                   index=False)
    header_written = True
    total_sampled += len(sampled)

    if total_sampled % 500_000 == 0:
        print(f"Sampled so far: {total_sampled:,}")

print(f"\nTotal sampled rows: {total_sampled:,}")
print(f"Saved to: {output_sampled}")

# Size check
size = os.path.getsize(output_sampled) / (1024*1024)
print(f"File size: {size:.1f} MB")

# Quick verify
df = pd.read_csv(output_sampled, nrows=5)
print(df)

Sampled so far: 500,000
Sampled so far: 1,000,000
Sampled so far: 1,500,000

Total sampled rows: 1,661,830
Saved to: /content/spark_data_processed/spark_logs_sampled.csv
File size: 250.4 MB
                           app_id                            container_id  \
0  application_1485248649253_0124  container_1485248649253_0124_02_000012   
1  application_1485248649253_0124  container_1485248649253_0124_02_000012   
2  application_1485248649253_0124  container_1485248649253_0124_02_000024   
3  application_1485248649253_0124  container_1485248649253_0124_02_000012   
4  application_1485248649253_0124  container_1485248649253_0124_02_000012   

                                            raw_line  
0  17/06/08 14:57:53 INFO executor.Executor: Fini...  
1  17/06/08 14:58:03 INFO executor.Executor: Fini...  
2  17/06/08 15:00:56 INFO executor.Executor: Fini...  
3  17/06/08 14:57:55 INFO executor.Executor: Runn...  
4  17/06/08 14:58:49 INFO executor.CoarseGrainedE...  


In [4]:
import pandas as pd
import os

sampled_path = "/content/spark_data_processed/spark_logs_sampled.csv"
output_labeled = "/content/spark_data_processed/spark_logs_labeled.csv"

CHUNK_SIZE = 100_000

# Keywords for anomaly detection
ERROR_KEYWORDS = ['ERROR', 'FATAL', 'Exception', 'FAILED', 'OutOfMemory',
                  'killed', 'Lost', 'Timeout', 'refused', 'unavailable']
WARN_KEYWORDS = ['WARN', 'WARNING']

header_written = False
total_anomaly = 0
total_normal = 0

for chunk in pd.read_csv(sampled_path, chunksize=CHUNK_SIZE, on_bad_lines='skip'):

    # Label banana
    def label_row(line):
        if any(kw in str(line) for kw in ERROR_KEYWORDS):
            return 1  # anomaly
        elif any(kw in str(line) for kw in WARN_KEYWORDS):
            return 2  # warning
        else:
            return 0  # normal

    chunk['label'] = chunk['raw_line'].apply(label_row)

    total_anomaly += (chunk['label'] == 1).sum()
    total_normal += (chunk['label'] == 0).sum()

    chunk.to_csv(output_labeled,
                 mode='a',
                 header=not header_written,
                 index=False)
    header_written = True

print(f"Labeling complete!")
print(f"Normal (0): {total_normal:,}")
print(f"Warning (2): {(1661830 - total_normal - total_anomaly):,}")
print(f"Anomaly (1): {total_anomaly:,}")
print(f"Saved: {output_labeled}")

Labeling complete!
Normal (0): 1,659,555
Warning (2): 242
Anomaly (1): 2,033
Saved: /content/spark_data_processed/spark_logs_labeled.csv


In [6]:
import pandas as pd
from drain3 import TemplateMiner

sampled_labeled_path = "/content/spark_data_processed/spark_logs_labeled.csv"
output_parsed = "/content/spark_data_processed/spark_logs_parsed.csv"

# Simple initialization - no config needed
miner = TemplateMiner()

CHUNK_SIZE = 100_000
header_written = False
total_processed = 0

for chunk in pd.read_csv(sampled_labeled_path,
                          chunksize=CHUNK_SIZE,
                          on_bad_lines='skip'):

    templates = []
    cluster_ids = []

    for line in chunk['raw_line'].astype(str):
        result = miner.add_log_message(line)
        templates.append(result['template_mined'])
        cluster_ids.append(result['cluster_id'])

    chunk['template'] = templates
    chunk['cluster_id'] = cluster_ids

    chunk.to_csv(output_parsed,
                 mode='a',
                 header=not header_written,
                 index=False)
    header_written = True
    total_processed += len(chunk)

    if total_processed % 500_000 == 0:
        print(f"Processed: {total_processed:,}")

print(f"\nDrain3 parsing complete!")
print(f"Total processed: {total_processed:,}")
print(f"Unique templates: {len(miner.drain.clusters)}")
print(f"Saved: {output_parsed}")

# Quick verify
df = pd.read_csv(output_parsed, nrows=3)
print(df[['raw_line', 'template', 'cluster_id', 'label']])

Processed: 500,000
Processed: 1,000,000
Processed: 1,500,000

Drain3 parsing complete!
Total processed: 1,661,830
Unique templates: 299
Saved: /content/spark_data_processed/spark_logs_parsed.csv
                                            raw_line  \
0  17/06/08 14:57:53 INFO executor.Executor: Fini...   
1  17/06/08 14:58:03 INFO executor.Executor: Fini...   
2  17/06/08 15:00:56 INFO executor.Executor: Fini...   

                                            template  cluster_id  label  
0  17/06/08 14:57:53 INFO executor.Executor: Fini...           1      0  
1  17/06/08 <*> INFO executor.Executor: Finished ...           1      0  
2  17/06/08 <*> INFO executor.Executor: Finished ...           1      0  


In [7]:
import pandas as pd

parsed_path = "/content/spark_data_processed/spark_logs_parsed.csv"

# Load sample for analysis
df = pd.read_csv(parsed_path, nrows=500_000)

print("=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"\nLabel Distribution:")
print(df['label'].value_counts())
print(f"\nUnique Apps: {df['app_id'].nunique()}")
print(f"Unique Containers: {df['container_id'].nunique()}")
print(f"Unique Templates: {df['cluster_id'].nunique()}")
print(f"\nTop 5 Templates:")
print(df['cluster_id'].value_counts().head())
print("\nColumns:", df.columns.tolist())
print("=" * 50)

FINAL DATASET SUMMARY
Shape: (500000, 6)

Label Distribution:
label
0    499255
1       647
2        98
Name: count, dtype: int64

Unique Apps: 90
Unique Containers: 1431
Unique Templates: 247

Top 5 Templates:
cluster_id
5    76274
3    63426
2    63408
1    63308
8    61920
Name: count, dtype: int64

Columns: ['app_id', 'container_id', 'raw_line', 'label', 'template', 'cluster_id']


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Drive pe folder banao
save_dir = "/content/drive/MyDrive/ASAD_Thesis/processed_data"
os.makedirs(save_dir, exist_ok=True)

# Files copy karo
files_to_save = [
    "/content/spark_data_processed/spark_logs_parsed.csv",
    "/content/spark_data_processed/spark_logs_labeled.csv",
    "/content/spark_data_processed/spark_logs_sampled.csv"
]

for f in files_to_save:
    filename = os.path.basename(f)
    dest = os.path.join(save_dir, filename)
    shutil.copy2(f, dest)
    size = os.path.getsize(dest) / (1024*1024)
    print(f"Saved: {filename} ({size:.1f} MB)")

print("\nAll files saved to Google Drive!")

Mounted at /content/drive
Saved: spark_logs_parsed.csv (365.1 MB)
Saved: spark_logs_labeled.csv (253.6 MB)
Saved: spark_logs_sampled.csv (250.4 MB)

All files saved to Google Drive!
